# Which of these warnings is real?—orbital conjunction screening

**Agents for Impact 2026 · the kickoff demo**

You operate a small constellation. There are roughly 32,000 objects in orbit with published positions,
your operations team gets a steady stream of close-approach warnings, and nearly all of them are noise.
**Which ones are real, and what do you do about them?**

There is a real trade underneath it. Moving a satellite costs propellant, and propellant is the
satellite's remaining life. Dodge everything and you end the mission early to avoid collisions that were
never going to happen. Dodge nothing and one day you become debris yourself.

**This notebook builds the data an agent needs to answer that honestly.** It loads the public orbital
catalogue into BigQuery, introduces our fictional operator—**Cymbal Orbital**, twelve satellites—and
screens every one of them against every real object for seven days. It ends where the agent begins.

> **Our satellites are made up. Everything they could hit is real.**

| Section | What happens | Kickoff segment it feeds |
|---|---|---|
| 1 | Setup | — |
| 2 | Where the data comes from, and loading it | "Your data is already loaded" |
| 3 | The wrong answer: an altitude filter | "Your data is already loaded" |
| 4 | Optional: prove the data is live | the opening beat |
| 5 | The real messes, and what we did about each | — |
| 6 | Into BigQuery | "Your data is already loaded" |
| 7 | What the data says: the hook, and Starlink | **the query that surprises them** |
| 8 | Meet Cymbal Orbital | — |
| 9 | The screen | — |
| 10 | The honest numbers | "Your one required differentiator" sets up from here |
| 11 | Validate before you build | — |
| 12 | Framing the agent | "Build an agent" |
| A | Maintainer only: publish the tables | — |
| B | Diagnostic summary | — |

**Time budget: about four minutes to run**, most of it the screen in section 9. Rather longer to read.

**Two ways through it.** *I've done this before:* Run all, then read sections 3, 7 and 10.
*I never have:* read the text between the cells as you go—it is written so that someone who runs nothing
can still follow every decision.

## 1—Setup

One install: `sgp4`, the standard implementation of the propagator every public orbital element set is
built for. It is new to this runtime, so installing it cannot collide with anything already loaded—and we
upgrade nothing else. (Upgrading a library Colab has already imported, without restarting, is how you get
errors that name nothing useful.)

Everything that could change the answer lives in the config cell, and every one of those values is
**pinned**: the snapshot, the fleet, the moment the screen starts. That is what lets the kickoff show the
same numbers in Toronto as in New York.

In [ ]:
import sys, subprocess, importlib.metadata as md
try:
    SGP4_VERSION = md.version("sgp4")
except md.PackageNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "sgp4"], check=True)
    SGP4_VERSION = md.version("sgp4")
print("sgp4", SGP4_VERSION)

In [ ]:
import os, io, json, math, time, hashlib, socket, platform, contextlib, datetime as dt
from io import StringIO
import numpy as np, pandas as pd
import google.auth
from google.cloud import bigquery, storage

# ---- what changes the answer: all pinned ----------------------------------------------------------
SNAPSHOT     = "20260925T0137Z"          # staged by notebooks/demo_90_stage_catalog.ipynb
SNAPSHOT_URI = f"gs://class-demo/a4i-2026/demo-orbital-conjunction/catalog/{SNAPSHOT}"
SCREEN_START = pd.Timestamp("2026-09-25T01:00:00Z")   # "now", for the whole demo
SCREEN_DAYS, STEP_S = 7, 60.0
FLEET_ALT_KM, PLANES, PER_PLANE = 880.0, 3, 4
HBR_M        = 5.0                       # combined hard-body radius—ASSUMED, see section 10
SIGMAS_M     = (200.0, 1000.0)           # assumed 1-sigma position uncertainty—ASSUMED
ESCALATE_PC  = 1e-4                      # our escalation line—a POLICY choice, see section 10
WATCH_M      = 1000.0                    # closer than this goes on the watch list
STALE_DAYS   = 5.0                       # elements older than this at closest approach: re-screen first

# ---- where things go -----------------------------------------------------------------------------
_, PROJECT = google.auth.default(); PROJECT = os.environ.get("GOOGLE_CLOUD_PROJECT") or PROJECT
DATASET  = "a4i_orbit"
LOCATION = "US"
PUBLISH  = False                         # maintainers only—see appendix A

MU, RE, J2 = 398600.4418, 6378.137, 1.08262668e-3     # WGS-72-era constants SGP4 is built on
BAND = (FLEET_ALT_KM - 25, FLEET_ALT_KM + 25)         # "an orbit that can reach our altitude"
bq = bigquery.Client(project=PROJECT)

TIMINGS, CHECKS, R = {}, [], {}
@contextlib.contextmanager
def step(name):
    t = time.perf_counter()
    try: yield
    finally: TIMINGS[name] = round(time.perf_counter() - t, 2); print(f"[{name}] {TIMINGS[name]}s")
def check(name, ok, detail=""):
    CHECKS.append(("PASS" if ok else "FAIL", name, str(detail))); print("PASS" if ok else "FAIL", "·", name, "·", detail)
def note(name, ok, detail=""):
    CHECKS.append(("OK" if ok else "WARN", name, str(detail))); print("OK" if ok else "WARN", "·", name, "·", detail)
def q(sql, **params):
    cfg = bigquery.QueryJobConfig(query_parameters=[
        bigquery.ScalarQueryParameter(k, "FLOAT64" if isinstance(v, float) else "INT64" if isinstance(v, int) else "STRING", v)
        for k, v in params.items()])
    return bq.query(sql, job_config=cfg).to_dataframe()

# The %%bigquery cell magic used in sections 7 and 9. Colab Enterprise usually has it already; if not, one of these loads it.
for ext in ("bigquery_magics", "google.cloud.bigquery"):
    try:
        get_ipython().run_line_magic("load_ext", ext); break
    except Exception as e:
        last = f"{ext}: {type(e).__name__}"
else:
    print("could not load the %%bigquery magic (", last, ")—the same queries run through q() in section 11")

R["fingerprint"] = {"python": sys.version.split()[0], "platform": platform.platform(), "host": socket.gethostname(),
                    "project": PROJECT, "vertex_product": os.environ.get("VERTEX_PRODUCT"),
                    "sgp4": SGP4_VERSION, "pandas": pd.__version__, "bigquery": md.version("google-cloud-bigquery"),
                    "run_utc": dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds")}
print(json.dumps(R["fingerprint"], indent=1))

## 2—Where the data comes from

Every tracked object in orbit is catalogued by **U.S. Space Command**, which publishes a set of
*mean orbital elements* for each one—six numbers and a timestamp that, fed to the right model, tell you
where the object will be. There are two front doors to that data:

- **[Space-Track.org](https://www.space-track.org)**, run for U.S. Space Command. Free, but you need an account.
  It has everything.
- **[CelesTrak](https://celestrak.org)**, which redistributes it with no account at all—but only in curated
  groups, and no query reaches the ~2,800 **dead satellites** in bulk.

Dead satellites matter more than anything else in this problem, because **they cannot get out of the way.**
So we pulled the full catalogue from Space-Track once, validated it, and staged it in Cloud Storage—that
is `notebooks/demo_90_stage_catalog.ipynb`, and re-running it is how the snapshot gets refreshed. The
satellite catalogue (SATCAT: what each object *is*—payload, rocket body, debris—who owns it, whether it
still works) comes from CelesTrak.

**What this data cannot do**, stated before we use it:

- **It has no covariance.** A real operator gets a statement of how uncertain each position is. We get a
  position. Every probability in this notebook therefore rests on an assumption we will say out loud.
- **Mean elements are not measurements.** The model reconstructs a position from them, and the error grows
  the further you are from the element set's timestamp—its *epoch*.
- **Objects nobody has updated in 30 days are not in the pull.** They are still catalogued; they are just
  lost. We count them in section 5.

**Rights, and the one obligation.** U.S. Space Command *"has provided express blanket approval for
transfer/redistribution of basic SSA data … conditioned on appropriate citation."* That covers orbital
elements and the satellite catalogue—exactly what we use. So:

> *Orbital data: U.S. Space Command, via Space-Track.org; satellite catalogue via CelesTrak (celestrak.org).*

In [ ]:
def gcs_text(uri):
    bkt, blob = uri[5:].split("/", 1)
    return storage.Client(project=PROJECT).bucket(bkt).blob(blob).download_as_text()

with step("load_snapshot"):
    MANIFEST = json.loads(gcs_text(f"{SNAPSHOT_URI}/manifest.json"))
    GP_TXT, SC_TXT = gcs_text(f"{SNAPSHOT_URI}/gp_full.csv"), gcs_text(f"{SNAPSHOT_URI}/satcat.csv")
    CITATION = gcs_text(f"{SNAPSHOT_URI}/CITATION.txt")

sha = lambda s: hashlib.sha256(s.encode()).hexdigest()
check("the catalogue is byte-for-byte the one the manifest describes", sha(GP_TXT) == MANIFEST["gp_full.csv"]["sha256"], SNAPSHOT)
check("the satellite catalogue matches its manifest too", sha(SC_TXT) == MANIFEST["satcat.csv"]["sha256"], SNAPSHOT)

GP  = pd.read_csv(StringIO(GP_TXT), dtype=str, keep_default_na=False)
SAT = pd.read_csv(StringIO(SC_TXT), dtype=str, keep_default_na=False)
print(f"orbital elements: {len(GP):,} objects, pulled {MANIFEST['gp_full.csv']['fetched_utc']} from Space-Track")
print(f"satellite catalogue: {len(SAT):,} rows (every object ever catalogued, including the {(SAT.DECAY_DATE != '').sum():,} that have re-entered)")
print("\n" + CITATION)

## 3—The wrong answer first

The obvious first move: an object can only hit you if it can reach your altitude. Our satellites fly at
880 km. Keep everything whose orbit—lowest point (*perigee*) to highest (*apogee*)—passes through 855–905 km.

Perigee and apogee come straight from two of the six elements: mean motion gives the orbit's size, and
eccentricity its shape.

In [ ]:
def add_orbit_cols(d):
    d = d.copy()
    n = pd.to_numeric(d["MEAN_MOTION"], errors="coerce"); e = pd.to_numeric(d["ECCENTRICITY"], errors="coerce")
    a = (MU / (n * 2*np.pi/86400.0)**2) ** (1/3)
    d["PERIGEE_KM"], d["APOGEE_KM"] = a*(1-e) - RE, a*(1+e) - RE
    return d
GP = add_orbit_cols(GP)
crossing = GP[(GP.PERIGEE_KM <= BAND[1]) & (GP.APOGEE_KM >= BAND[0])]
R["wrong_answer"] = {"objects_crossing_our_altitude": len(crossing), "times_12_satellites": 12 * len(crossing)}
print(f"{len(crossing):,} objects have orbits that pass through our altitude.")
print(f"Twelve satellites, so by this test we have {12*len(crossing):,} things to worry about—every week, forever.")

**Every one of those objects really will be at our altitude—twice an orbit, about fourteen orbits a day.** And
almost none of them will ever be anywhere near us, because being at the same *height* is not being at the
same *place*. Two orbits at 880 km cross at two points in space, and a collision needs both objects at
the same one of those points **in the same second**, closing at up to 15 km/s.

So altitude is a fine first filter—it is exactly the prefilter the screen uses in section 9—and a useless
last one. The real question needs time and geometry: *where will each object be, second by second, for
the next week, and how close does it come?* Hold on to that number; section 9 answers it properly, and the
difference between the two is the premise of the whole demo.

## 4—Optional: prove the data is live

The analysis uses the frozen snapshot, so the numbers on screen are the same in every city. This cell
asks CelesTrak, live, for one object—the International Space Station—just to show that the data is
current and public. **If it fails for any reason it says so and moves on.** It never retries: CelesTrak
asks clients to stop on any error, and firewalls an address after 50 of them.

A second run inside two hours gets a polite HTTP 403—*"GP data has not updated since your last successful
download"*—which is itself proof the service is doing its job.

In [ ]:
import requests
try:
    r = requests.get("https://celestrak.org/NORAD/elements/gp.php?CATNR=25544&FORMAT=csv",
                     headers={"User-Agent": "A4I2026-demo/1.0 (ROI Training)"}, timeout=10)
    if r.status_code == 200 and r.text.startswith("OBJECT_NAME"):
        iss = pd.read_csv(StringIO(r.text), dtype=str).iloc[0]
        age_h = (pd.Timestamp.now(tz="UTC") - pd.to_datetime(iss.EPOCH, utc=True, format="ISO8601")).total_seconds() / 3600
        R["live_beat"] = f"ISS elements {age_h:.1f} h old"
        print(f"{iss.OBJECT_NAME}: elements published {age_h:.1f} hours ago.")
    else:
        R["live_beat"] = f"HTTP {r.status_code}: {r.text[:90]!r}"
        print("CelesTrak said:", r.status_code, r.text[:160])
except Exception as e:
    R["live_beat"] = f"skipped: {type(e).__name__}"
    print("Live check skipped:", type(e).__name__, "—the snapshot is all the demo needs.")

## 5—The real messes, and what we did about each

Public orbital data is excellent and it is still data. Here is everything we tripped over, in the order it
would trip you.

### 5.1 Six-digit catalogue numbers broke the classic format

For sixty years orbital elements travelled as the **two-line element set (TLE)**, with a five-character
field for the catalogue number. The catalogue passed 99,999 this summer, and CelesTrak ran out of
five-digit numbers on 2026-07-11. So we use **OMM**, the modern field-by-field format, and never the TLE
columns Space-Track still ships. Look at what those columns hold for the newest objects:

In [ ]:
GP["NORAD_INT"] = pd.to_numeric(GP["NORAD_CAT_ID"])
six = GP[GP.NORAD_INT >= 100000]
print(f"{len(six):,} objects have six-digit catalogue numbers. What Space-Track put in the TLE columns for three of them:\n")
for _, x in six.head(3).iterrows():
    print(f"  {x.NORAD_CAT_ID:>7}  {x.OBJECT_NAME:<22} TLE_LINE1 = {x.get('TLE_LINE1', '(no column)')!r}")
R["six_digit"] = len(six)

Whatever encoding you see there is a workaround, and a workaround has to be understood by every tool that
reads it. OMM carries the number as a plain integer. **Rule: read OMM, drop the TLE lines.**

### 5.2 The timestamp format is stricter than it looks

`sgp4` reads each element set's epoch with one exact format—fractional seconds required. And pandas 3
guesses a date format from the **first** row: give it one timestamp without fractional seconds and it turns
**every other row** into "not a time," silently. Watch:

In [ ]:
demo_epochs = pd.Series(["2026-09-24T00:06:10", "2026-09-24T03:48:32.657184", "2026-09-24T01:46:04.721664"])
print("pandas, guessing: ", pd.to_datetime(demo_epochs, utc=True, errors="coerce").isna().sum(), "of 3 became NaT")
print("pandas, told ISO: ", pd.to_datetime(demo_epochs, utc=True, errors="coerce", format="ISO8601").isna().sum(), "of 3 became NaT")

GP["EPOCH_TS"] = pd.to_datetime(GP["EPOCH"], utc=True, errors="coerce", format="ISO8601")
check("every epoch in the catalogue parses", GP.EPOCH_TS.notna().all(), int(GP.EPOCH_TS.isna().sum()))

### 5.3 Elements age, and age is uncertainty

An element set is most accurate at its epoch and degrades from there. So "how old are these elements *at
the moment of closest approach*" is a real quality signal, and we carry it all the way into the
conjunctions table.

In [ ]:
GP["ELEMENT_AGE_DAYS"] = (SCREEN_START - GP.EPOCH_TS).dt.total_seconds() / 86400
ages = GP.ELEMENT_AGE_DAYS.quantile([0.1, 0.5, 0.9, 0.99, 1.0]).round(2)
R["element_age_days"] = ages.to_dict()
print("element age at the start of the screen, in days:"); print(ages.to_string())
note("no element set is older than the 30-day pull window", GP.ELEMENT_AGE_DAYS.max() <= 31, round(GP.ELEMENT_AGE_DAYS.max(), 2))

### 5.4 SGP4 is a near-Earth model, and it says so

Every row must initialise in the propagator, or the screen silently skips it. Two rows in this snapshot
will not—and they are the right two to fail:

In [ ]:
from sgp4.api import Satrec, SatrecArray, WGS72, jday
from sgp4 import omm
OMM = ["OBJECT_NAME","OBJECT_ID","EPOCH","MEAN_MOTION","ECCENTRICITY","INCLINATION","RA_OF_ASC_NODE","ARG_OF_PERICENTER",
       "MEAN_ANOMALY","EPHEMERIS_TYPE","CLASSIFICATION_TYPE","NORAD_CAT_ID","ELEMENT_SET_NO","REV_AT_EPOCH","BSTAR",
       "MEAN_MOTION_DOT","MEAN_MOTION_DDOT"]

def omm_fields(rec):
    # sgp4 parses EPOCH with '%Y-%m-%dT%H:%M:%S.%f' and int()s two counters: normalise both before it sees them
    f = {k: str(rec[k]) for k in OMM}
    if "." not in f["EPOCH"]: f["EPOCH"] += ".000000"
    for k in ("ELEMENT_SET_NO", "REV_AT_EPOCH", "EPHEMERIS_TYPE"):
        f[k] = f[k] if f[k].strip().lstrip("-").isdigit() else "0"
    f["CLASSIFICATION_TYPE"] = f["CLASSIFICATION_TYPE"] or "U"
    return f

def satrec(rec):
    s = Satrec(); omm.initialize(s, omm_fields(rec)); return s

ok = []
for rec in GP[OMM].to_dict("records"):
    try: ok.append(satrec(rec).error == 0)
    except Exception: ok.append(False)
GP["SGP4_OK"] = ok
bad = GP[~GP.SGP4_OK]
print(bad[["NORAD_CAT_ID", "OBJECT_NAME", "OBJECT_ID", "PERIGEE_KM", "APOGEE_KM"]].to_string(index=False))
note("every object initialises in SGP4", len(bad) == 0, f"{len(bad)} cannot: {bad.OBJECT_NAME.tolist()}")
check("almost every object initialises in SGP4", len(bad) <= 0.001 * len(GP), f"{len(bad)} of {len(GP):,}")

Those are, in this snapshot, the Nancy Grace Roman Space Telescope and the Falcon Heavy stage that
launched it, on their way to deep space. SGP4 was built for objects that stay near Earth and it refuses the
orbit rather than inventing a position. **An error that names its reason is the good kind.** Neither is
anywhere near 880 km.

### 5.5 "What is this object?" has two answers—use the right one

Space-Track's elements carry an `OBJECT_TYPE`. So does SATCAT. They disagree, because Space-Track labels
newly launched pieces `UNKNOWN` until they are identified:

In [ ]:
print("Space-Track OBJECT_TYPE:", GP["OBJECT_TYPE"].value_counts().to_dict() if "OBJECT_TYPE" in GP else "absent")
sc_on = SAT[(SAT.ORBIT_CENTER == "EA") & (SAT.DECAY_DATE == "")]
print("SATCAT OBJECT_TYPE (on orbit):", sc_on.OBJECT_TYPE.value_counts().to_dict())

We take the type, the owner, the launch date and the **operational status** from SATCAT—operational
status is the one that matters most, because an operational satellite might dodge too, and a dead one
never will.

### 5.6 Some tracked objects are not in the pull at all

In [ ]:
sc_el = sc_on[sc_on.DATA_STATUS_CODE != "NEA"]
lost = sc_el[~sc_el.NORAD_CAT_ID.isin(set(GP.NORAD_CAT_ID))]
R["not_in_pull"] = {"count": len(lost), "by_type": lost.OBJECT_TYPE.value_counts().to_dict()}
print(f"{len(lost):,} of {len(sc_el):,} catalogued on-orbit objects have no element set newer than 30 days:", R["not_in_pull"]["by_type"])
note("the pull covers the catalogue", len(lost) / len(sc_el) < 0.1, f"{len(lost):,} lost objects are invisible to this screen")

These are mostly debris the tracking network has not updated in over a month—**lost**, in the catalogue's
own vocabulary. We cannot screen against a position nobody has measured recently, and we will not pretend
to. It goes on the list of things the agent must say it cannot see.

### 5.7 Which pieces came from 2007 and 2009

We attribute debris to its parent event by **name** (`FENGYUN 1C DEB`, `IRIDIUM 33 DEB`, `COSMOS 2251 DEB`),
not by launch designator. The designator would also sweep in each parent's rocket body, and—for the
Iridium launch—four other satellites that never broke up.

In [ ]:
EVENTS = {"FENGYUN 1C DEB": ("2007 Chinese anti-satellite test", "FENGYUN 1C"),
          "IRIDIUM 33 DEB": ("2009 Iridium–Cosmos collision", "IRIDIUM 33"),
          "COSMOS 2251 DEB": ("2009 Iridium–Cosmos collision", "COSMOS 2251")}
def parent(name):
    for k, v in EVENTS.items():
        if name.startswith(k): return v
    return (None, None)

# SATCAT: typed, with the two derived columns every later query uses
SAT["PARENT_EVENT"], SAT["PARENT_OBJECT"] = zip(*SAT.OBJECT_NAME.map(parent))
SAT["ON_ORBIT"] = (SAT.ORBIT_CENTER == "EA") & (SAT.DECAY_DATE == "")
SAT["IS_STARLINK"] = SAT.OBJECT_NAME.str.startswith("STARLINK")
for src, dst in [("PERIOD", "PERIOD_MIN"), ("INCLINATION", "INCLINATION_DEG"), ("APOGEE", "APOGEE_KM"),
                 ("PERIGEE", "PERIGEE_KM"), ("RCS", "RCS_M2")]:
    SAT[dst] = pd.to_numeric(SAT[src], errors="coerce")
SAT["NORAD_CAT_ID"] = pd.to_numeric(SAT.NORAD_CAT_ID).astype("Int64")
for c in ("LAUNCH_DATE", "DECAY_DATE"):              # DATE in BigQuery; missing stays missing, never the string "NaT"
    d = pd.to_datetime(SAT[c], errors="coerce", format="ISO8601")
    SAT[c] = d.dt.date.astype(object).where(d.notna(), None)
for c in ("OPS_STATUS_CODE", "OWNER", "RCS", "DATA_STATUS_CODE"):
    SAT[c] = SAT[c].fillna("")
SATCAT = SAT[["OBJECT_NAME","OBJECT_ID","NORAD_CAT_ID","OBJECT_TYPE","OPS_STATUS_CODE","OWNER","LAUNCH_DATE","LAUNCH_SITE",
              "DECAY_DATE","PERIOD_MIN","INCLINATION_DEG","APOGEE_KM","PERIGEE_KM","RCS_M2","DATA_STATUS_CODE","ORBIT_CENTER",
              "ORBIT_TYPE","ON_ORBIT","IS_STARLINK","PARENT_EVENT","PARENT_OBJECT"]].copy()
print(SATCAT[SATCAT.PARENT_EVENT.notna()].groupby(["PARENT_OBJECT", "ON_ORBIT"]).size().unstack(fill_value=0)
      .rename(columns={True: "still in orbit", False: "re-entered"}).to_string())

### 5.8 Two ways to compute perigee—do they agree?

Space-Track publishes `PERIAPSIS` and `APOAPSIS`. We derived our own from mean motion and eccentricity.
They should agree to within the few kilometres that separate slightly different Earth-radius and
gravity constants; if they did not, one of us would have a units problem.

In [ ]:
if "PERIAPSIS" in GP.columns:
    diff = (pd.to_numeric(GP.PERIAPSIS, errors="coerce") - GP.PERIGEE_KM).abs()
    R["perigee_agreement_km"] = {"median": round(float(diff.median()), 3), "p99": round(float(diff.quantile(0.99)), 3)}
    print("our perigee vs Space-Track's, |difference| in km:", R["perigee_agreement_km"])
    check("our derived perigee agrees with Space-Track's", diff.median() < 5, R["perigee_agreement_km"])

## 6—Into BigQuery

Five tables in `a4i_orbit`, in your project. The OMM-shaped tables keep the standard OMM field names in
capitals, so any OMM-aware tool reads them as-is; our own analysis tables use lower-case names.

| Table | Grain | What it is |
|---|---|---|
| `catalog` | one row per object with current elements | OMM fields + perigee, apogee, element age, and SATCAT's type, status, owner and parent event |
| `satcat` | one row per object ever catalogued | what each object is, including the ones that have re-entered |
| `fleet` | one row per Cymbal Orbital satellite | our twelve fictional satellites, in the same OMM shape as `catalog` |
| `conjunctions` | one row per close approach | the screen's output—section 9 |
| `snapshot_info` | one row | which snapshot, which moment, which assumptions |

Every load replaces the table, so this is safe to re-run.

In [ ]:
look = SAT.set_index(pd.to_numeric(SAT.NORAD_CAT_ID).astype("Int64"))
dup_ids = look.index.duplicated()
check("SATCAT has one row per catalogue number", not dup_ids.any(), int(dup_ids.sum()))
look = look[~dup_ids]
CAT = GP[OMM + ["EPOCH_TS", "PERIGEE_KM", "APOGEE_KM", "ELEMENT_AGE_DAYS", "SGP4_OK"]
         + [c for c in ("RCS_SIZE",) if c in GP.columns]].copy()
CAT["NORAD_CAT_ID"] = pd.to_numeric(CAT.NORAD_CAT_ID).astype("Int64")
for c in ("MEAN_MOTION","ECCENTRICITY","INCLINATION","RA_OF_ASC_NODE","ARG_OF_PERICENTER","MEAN_ANOMALY","BSTAR",
          "MEAN_MOTION_DOT","MEAN_MOTION_DDOT"):
    CAT[c] = pd.to_numeric(CAT[c], errors="coerce")
for c in ("EPHEMERIS_TYPE", "ELEMENT_SET_NO", "REV_AT_EPOCH"):
    CAT[c] = pd.to_numeric(CAT[c], errors="coerce").astype("Int64")
for src, dst in [("OBJECT_TYPE", "OBJECT_TYPE"), ("OPS_STATUS_CODE", "OPS_STATUS_CODE"), ("OWNER", "OWNER"),
                 ("LAUNCH_DATE", "LAUNCH_DATE"), ("PARENT_EVENT", "PARENT_EVENT"), ("IS_STARLINK", "IS_STARLINK")]:
    CAT[dst] = CAT.NORAD_CAT_ID.map(look[src])
CAT["OBJECT_TYPE"] = CAT.OBJECT_TYPE.fillna("UNK")
CAT["OPS_STATUS_CODE"] = CAT.OPS_STATUS_CODE.fillna(""); CAT["OWNER"] = CAT.OWNER.fillna("")
CAT["IS_STARLINK"] = CAT.IS_STARLINK.fillna(False).astype(bool)
CAT["PARENT_EVENT"] = CAT.PARENT_EVENT.astype(object).where(CAT.PARENT_EVENT.notna(), None)
joined = CAT.NORAD_CAT_ID.isin(look.index).sum()
check("every catalogue object found in SATCAT", joined == len(CAT), f"{joined:,} of {len(CAT):,}")

def load(df, table):
    job = bq.load_table_from_dataframe(df, f"{PROJECT}.{DATASET}.{table}",
                                       job_config=bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE"))
    job.result(); return bq.get_table(f"{PROJECT}.{DATASET}.{table}").num_rows

with step("bq_load"):
    ds = bigquery.Dataset(f"{PROJECT}.{DATASET}"); ds.location = LOCATION
    bq.create_dataset(ds, exists_ok=True)
    n_cat, n_sat = load(CAT, "catalog"), load(SATCAT, "satcat")
print(f"catalog {n_cat:,} rows · satcat {n_sat:,} rows")

## 7—What the data says

### The query that surprises people

A big share of what is up there traces back to two events. In **January 2007** China destroyed its own
dead weather satellite, Fengyun-1C, with a missile. In **February 2009** a working Iridium satellite and a
dead Russian one, Cosmos 2251, collided at about 11.7 km/s. How much of what crosses *our* altitude came
from those two moments?

In [ ]:
%%bigquery hook
-- Everything catalogued and still in orbit whose path crosses 855–905 km: our fleet's altitude, ±25 km
SELECT
  COUNT(*)                                                       AS objects_crossing_880_km,
  COUNTIF(PARENT_EVENT IS NOT NULL)                              AS from_2007_and_2009,
  ROUND(100 * COUNTIF(PARENT_EVENT IS NOT NULL) / COUNT(*), 1)   AS pct_of_everything,
  ROUND(100 * COUNTIF(PARENT_EVENT IS NOT NULL)
            / COUNTIF(OBJECT_TYPE = 'DEB'), 1)                   AS pct_of_the_debris
FROM a4i_orbit.satcat
WHERE ON_ORBIT AND PERIGEE_KM <= 905 AND APOGEE_KM >= 855

In [ ]:
%%bigquery still_up
SELECT PARENT_OBJECT, PARENT_EVENT,
       COUNTIF(ON_ORBIT)     AS still_in_orbit,
       COUNTIF(NOT ON_ORBIT) AS already_reentered
FROM a4i_orbit.satcat
WHERE PARENT_EVENT IS NOT NULL
GROUP BY PARENT_OBJECT, PARENT_EVENT
ORDER BY still_in_orbit DESC

**Kessler syndrome in one table.** A collision makes debris, debris makes more collisions, and at this
altitude there is very little air to drag anything down—so seventeen years after Iridium 33 met Cosmos
2251, hundreds of their pieces are still up there, and still generating warnings.

### And the other thing everyone asks about: Starlink

In [ ]:
%%bigquery starlink
SELECT
  COUNT(*)                                                                         AS starlink_in_orbit,
  ROUND(100 * COUNT(*) / (SELECT COUNT(*) FROM a4i_orbit.satcat WHERE ON_ORBIT), 1) AS pct_of_everything_in_orbit,
  APPROX_QUANTILES((PERIGEE_KM + APOGEE_KM) / 2, 100)[OFFSET(50)]                  AS median_altitude_km,
  COUNTIF(PERIGEE_KM <= 905 AND APOGEE_KM >= 855)                                  AS crossing_880_km
FROM a4i_orbit.satcat
WHERE ON_ORBIT AND IS_STARLINK

**About a third of everything in orbit is now Starlink—and none of it comes near us.** That splits the
picture in two. The new crowding is low, around 470 km, where the thin atmosphere pulls dead hardware down
within a few years. The old debris is up here, where it stays for decades. We are fictional; the
neighbourhood is not.

In [ ]:
# The same three answers computed in pandas, so a %%bigquery cell that did not run cannot hide a wrong number
on = SATCAT[SATCAT.ON_ORBIT]; band = on[(on.PERIGEE_KM <= BAND[1]) & (on.APOGEE_KM >= BAND[0])]
sl = on[on.IS_STARLINK]
R["hook"] = {"objects_crossing": len(band), "from_2007_2009": int(band.PARENT_EVENT.notna().sum()),
             "pct_of_everything": round(100 * band.PARENT_EVENT.notna().sum() / len(band), 1),
             "pct_of_debris": round(100 * band.PARENT_EVENT.notna().sum() / (band.OBJECT_TYPE == "DEB").sum(), 1),
             "global_pct_of_everything": round(100 * on.PARENT_EVENT.notna().sum() / len(on), 1),
             "still_in_orbit": on.groupby("PARENT_OBJECT").size().to_dict()}
R["starlink"] = {"in_orbit": len(sl), "pct_of_everything": round(100 * len(sl) / len(on), 1),
                 "median_alt_km": round(float(((sl.PERIGEE_KM + sl.APOGEE_KM) / 2).median())) if len(sl) else None,
                 "crossing_our_altitude": int(((sl.PERIGEE_KM <= BAND[1]) & (sl.APOGEE_KM >= BAND[0])).sum())}
print(json.dumps({"hook": R["hook"], "starlink": R["starlink"]}, indent=1))
if "hook" in globals() and isinstance(globals()["hook"], pd.DataFrame):
    check("BigQuery and pandas agree on the hook", int(hook.iloc[0].from_2007_and_2009) == R["hook"]["from_2007_2009"],
          f"sql {int(hook.iloc[0].from_2007_and_2009)} · pandas {R['hook']['from_2007_2009']}")

## 8—Meet Cymbal Orbital

Cymbal Orbital is fictional: twelve small Earth-observation satellites in three orbital planes of four,
**sun-synchronous** at 880 km—the orbit Earth-imaging satellites favour, because the orbit's slow drift
keeps pace with the Sun and every pass over a given place happens at the same local time.

Sun-synchronous is not a choice of inclination so much as a consequence of altitude. Earth's equatorial
bulge makes an inclined orbit's plane rotate; at exactly the right tilt for a given height, it rotates once
a year. The cell below computes that tilt rather than looking it up.

**Every value here is pinned**, because the conjunctions this fleet meets are a property of exactly this
fleet at exactly this moment. Change one digit and the demo's close approaches change with it. The
catalogue numbers are 270001–270012—deliberately far above any real one (the real catalogue is just past
100,800).

In [ ]:
def sso_inclination(alt_km):
    a = RE + alt_km; n = math.sqrt(MU / a**3); rate = 2*math.pi / (365.2422 * 86400)
    return math.degrees(math.acos(-rate * 2 * a**2 / (3 * J2 * RE**2 * n)))

INC = sso_inclination(FLEET_ALT_KM)
MEAN_MOTION_REV_DAY = math.sqrt(MU / (RE + FLEET_ALT_KM)**3) * 86400 / (2*math.pi)
rows = []
for p in range(PLANES):
    for s in range(PER_PLANE):
        k = p * PER_PLANE + s + 1
        rows.append({"OBJECT_NAME": f"CYMBAL-{k:02d}", "OBJECT_ID": "FICTIONAL", "EPOCH": SCREEN_START.strftime("%Y-%m-%dT%H:%M:%S.%f"),
                     "MEAN_MOTION": MEAN_MOTION_REV_DAY, "ECCENTRICITY": 0.0005, "INCLINATION": INC,
                     "RA_OF_ASC_NODE": 360.0 * p / PLANES, "ARG_OF_PERICENTER": 90.0,
                     "MEAN_ANOMALY": 360.0 * s / PER_PLANE + (180.0 / PER_PLANE) * (p % 2),
                     "EPHEMERIS_TYPE": 0, "CLASSIFICATION_TYPE": "U", "NORAD_CAT_ID": 270000 + k,
                     "ELEMENT_SET_NO": 1, "REV_AT_EPOCH": 0, "BSTAR": 2e-5, "MEAN_MOTION_DOT": 0.0, "MEAN_MOTION_DDOT": 0.0,
                     "OPERATOR": "Cymbal Orbital (fictional)", "PLANE": p + 1, "SLOT": s + 1})
FLEET = add_orbit_cols(pd.DataFrame(rows))
FLEET_SATS = [satrec(r) for r in FLEET[OMM].to_dict("records")]
check("every Cymbal satellite initialises", all(s.error == 0 for s in FLEET_SATS), len(FLEET_SATS))
print(f"inclination {INC:.3f}°, {MEAN_MOTION_REV_DAY:.4f} orbits a day, perigee–apogee "
      f"{FLEET.PERIGEE_KM.min():.1f}–{FLEET.APOGEE_KM.max():.1f} km")
print(FLEET[["OBJECT_NAME", "PLANE", "SLOT", "RA_OF_ASC_NODE", "MEAN_ANOMALY"]].to_string(index=False))
n_fleet = load(FLEET, "fleet")

## 9—The screen

Now the real question: for every Cymbal satellite and every catalogued object that can reach our
altitude, **how close do they come, and when, over the next seven days?**

1. **Prefilter** to the orbits that pass through 855–905 km—section 3's "wrong answer," now doing the job it
   is actually good for.
2. **Propagate** all of them, and our twelve, every 60 seconds for a week with SGP4.
3. At each step, treat the relative motion as a straight line for that one minute—at closing speeds of
   5–15 km/s the paths really are straight over a minute—and compute where the closest approach falls.
4. **Refine** every approach under 10 km against SGP4 itself, to the millisecond.

About two minutes. The heavy lifting runs here, in the notebook, on purpose: the agent's sandbox is for the
*what-if* questions an operator asks about one approach, not for re-screening the sky.

In [ ]:
JD0, FR0 = jday(SCREEN_START.year, SCREEN_START.month, SCREEN_START.day, SCREEN_START.hour, SCREEN_START.minute, SCREEN_START.second)
def pos_at(s, ts):
    e, r, v = s.sgp4(JD0, FR0 + ts / 86400.0); return np.array(r), np.array(v)

def refine(fs, cs, t0):
    a, b = t0 - 5.0, t0 + 5.0; g = (math.sqrt(5) - 1) / 2
    d = lambda t: np.linalg.norm(pos_at(fs, t)[0] - pos_at(cs, t)[0])
    x1, x2 = b - g*(b-a), a + g*(b-a); f1, f2 = d(x1), d(x2)
    for _ in range(40):
        if f1 < f2: b, x2, f2 = x2, x1, f1; x1 = b - g*(b-a); f1 = d(x1)
        else:       a, x1, f1 = x1, x2, f2; x2 = a + g*(b-a); f2 = d(x2)
    return (a + b) / 2

def screen(fleet, cat, report_km=10.0, chunk=720):
    nsteps = int(SCREEN_DAYS * 86400 / STEP_S); hits = []
    for c0 in range(0, nsteps, chunk):
        k = np.arange(c0, min(nsteps, c0 + chunk)); fr = FR0 + k * STEP_S / 86400.0; jd = np.full(k.shape, JD0)
        ef, rf, vf = SatrecArray(fleet).sgp4(jd, fr); ec, rc, vc = SatrecArray(cat).sgp4(jd, fr)
        for i in range(len(fleet)):
            dr, dv = rc - rf[i][None], vc - vf[i][None]
            vv = np.einsum("ctk,ctk->ct", dv, dv)
            t = np.clip(-np.einsum("ctk,ctk->ct", dr, dv) / np.maximum(vv, 1e-12), -STEP_S/2, STEP_S/2)
            m = np.linalg.norm(dr + dv * t[..., None], axis=2); m[ec != 0] = np.inf
            hits += [(i, int(c), float(k[tt] * STEP_S + t[c, tt])) for c, tt in zip(*np.nonzero(m < report_km))]
    out = {}
    for i, c, t0 in hits:                       # refine, keeping one row per encounter (same pair within 2 minutes)
        ts = refine(fleet[i], cat[c], t0)
        key = (i, c, round(ts / 120))
        r1, v1 = pos_at(fleet[i], ts); r2, v2 = pos_at(cat[c], ts)
        miss = float(np.linalg.norm(r2 - r1))
        if key not in out or miss < out[key]["miss_km"]:
            out[key] = {"i": i, "c": c, "t_s": ts, "miss_km": miss, "r1": r1, "v1": v1, "r2": r2, "v2": v2}
    return list(out.values())

with step("screen"):
    cand = CAT[CAT.SGP4_OK & (CAT.PERIGEE_KM <= BAND[1]) & (CAT.APOGEE_KM >= BAND[0])].reset_index(drop=True)
    CAND_SATS = [satrec(r) for r in cand[OMM].to_dict("records")]
    EVENTS_RAW = screen(FLEET_SATS, CAND_SATS)
print(f"{len(cand):,} candidates · {len(EVENTS_RAW):,} approaches under 10 km in {SCREEN_DAYS} days")

### Turning approaches into rows an operator can use

For each approach we record the miss distance and also **how it splits into radial, in-track and
cross-track**—up/down, ahead/behind, and sideways from our satellite's point of view. That split is what
an operator plans a manoeuvre against: a burn along the direction of travel moves you in-track, so an
approach that is mostly in-track is the easiest kind to fix.

Then the risk, which needs the assumptions from section 1. **Public elements carry no covariance**, so a
true probability of collision is not available. We report two honest things instead:

- **Maximum possible probability.** Whatever the uncertainty actually is, the probability cannot exceed
  this. It needs no assumption about covariance—only about size.
- **Probability if the 1-sigma uncertainty is 200 m, and if it is 1 km.** Stated, not hidden, so you can
  see how far the answer moves when the assumption does.

Both treat the uncertainty as the same in every direction, and both need a **hard-body radius**—how big the
two objects are together. We assume 5 m.

In [ ]:
def max_pc(d_m, hbr=HBR_M):            # worst case over every isotropic 1-sigma, small-object form
    return min(1.0, hbr**2 / (math.e * d_m**2)) if d_m > 3 * hbr else 1.0
def pc_sigma(d_m, sigma, hbr=HBR_M):   # isotropic 2-D Gaussian in the encounter plane, small-object form
    return (hbr**2 / (2 * sigma**2)) * math.exp(-d_m**2 / (2 * sigma**2))

rows = []
for h in EVENTS_RAW:
    o = cand.iloc[h["c"]]; f = FLEET.iloc[h["i"]]
    R_hat = h["r1"] / np.linalg.norm(h["r1"]); C_hat = np.cross(h["r1"], h["v1"]); C_hat /= np.linalg.norm(C_hat)
    I_hat = np.cross(C_hat, R_hat); d = (h["r2"] - h["r1"]) * 1000
    # BigQuery TIMESTAMP holds microseconds; a float of seconds gives pandas nanoseconds, which the load refuses to truncate
    tca = (SCREEN_START + pd.Timedelta(seconds=h["t_s"])).round("us"); miss_m = h["miss_km"] * 1000
    age = (tca - o.EPOCH_TS).total_seconds() / 86400
    rows.append({"fleet_sat": f.OBJECT_NAME, "norad_cat_id": int(o.NORAD_CAT_ID), "object_name": o.OBJECT_NAME,
                 "object_id": o.OBJECT_ID, "object_type": o.OBJECT_TYPE, "ops_status": o.OPS_STATUS_CODE or "",
                 "parent_event": o.PARENT_EVENT, "tca_utc": tca, "hours_from_now": round(h["t_s"] / 3600, 2),
                 "miss_m": round(miss_m, 1), "radial_m": round(float(d @ R_hat), 1), "in_track_m": round(float(d @ I_hat), 1),
                 "cross_track_m": round(float(d @ C_hat), 1),
                 "rel_speed_km_s": round(float(np.linalg.norm(h["v2"] - h["v1"])), 2),
                 "element_age_at_tca_days": round(age, 2), "hbr_m": HBR_M,
                 "max_pc": max_pc(miss_m), "pc_sigma_200m": pc_sigma(miss_m, SIGMAS_M[0]), "pc_sigma_1km": pc_sigma(miss_m, SIGMAS_M[1])})
CONJ = pd.DataFrame(rows).sort_values("max_pc", ascending=False).reset_index(drop=True)
CONJ["elements_stale"] = CONJ.element_age_at_tca_days > STALE_DAYS
CONJ["triage"] = np.where(CONJ.max_pc >= ESCALATE_PC, "ESCALATE", np.where(CONJ.miss_m < WATCH_M, "WATCH", "NOISE"))
CONJ["parent_event"] = CONJ.parent_event.astype(object).where(CONJ.parent_event.notna(), None)
n_conj = load(CONJ, "conjunctions")
R["screen"] = {"candidates": len(cand), "lt10km": len(CONJ), "lt5km": int((CONJ.miss_m < 5000).sum()),
               "lt1km": int((CONJ.miss_m < 1000).sum()), "triage": CONJ.triage.value_counts().to_dict()}
print(json.dumps(R["screen"], indent=1))

In [ ]:
%%bigquery top
SELECT fleet_sat, object_name, object_type, parent_event,
       FORMAT_TIMESTAMP('%a %d %b %H:%M UTC', tca_utc) AS closest_approach,
       miss_m, rel_speed_km_s, element_age_at_tca_days AS elements_days_old, max_pc, triage
FROM a4i_orbit.conjunctions
ORDER BY max_pc DESC
LIMIT 10

## 10—The honest numbers

Section 3 said every one of those objects was a threat. Here is the week, measured:

In [ ]:
print(f"objects that cross our altitude (section 3):   {len(cand):>6,}")
print(f"approaches under 10 km in the next {SCREEN_DAYS} days:    {R['screen']['lt10km']:>6,}")
print(f"under 5 km:                                    {R['screen']['lt5km']:>6,}")
print(f"under 1 km—the watch list:                     {R['screen']['lt1km']:>6,}")
print(f"worst-case probability over 1 in {1/ESCALATE_PC:,.0f}:           {int((CONJ.triage == 'ESCALATE').sum()):>6,}")

**Nearly all of it is noise, and the screen is how you know which part isn't.** Two approaches this week
are worth a human's attention, and they are worth attention for different reasons:

In [ ]:
PINNED = {"A": ("CYMBAL-04", 31178, 301.1), "B": ("CYMBAL-11", 8520, 420.7)}   # from the Stage 1 probe, same snapshot
show = ["fleet_sat","object_name","object_type","parent_event","tca_utc","hours_from_now","miss_m","radial_m","in_track_m",
        "cross_track_m","rel_speed_km_s","element_age_at_tca_days","elements_stale","max_pc","pc_sigma_200m","pc_sigma_1km","triage"]
for label, (sat, norad, miss) in PINNED.items():
    hit = CONJ[(CONJ.fleet_sat == sat) & (CONJ.norad_cat_id == norad)]
    if SNAPSHOT == "20260925T0137Z":
        check(f"moment {label} reproduced: {sat} vs {norad} at {miss} m", len(hit) == 1 and abs(hit.miss_m.min() - miss) < 1.0,
              hit.miss_m.tolist())
    if len(hit):
        print(f"\n— Moment {label} —"); print(hit[show].T.to_string(header=False))
        R[f"moment_{label}"] = {k: (v.isoformat() if isinstance(v, pd.Timestamp) else v.item() if hasattr(v, "item") else v)
                                for k, v in hit.iloc[0][show].items()}

**Moment A—the closest approach, and the hook.** Nineteen hours from now, CYMBAL-04 passes about **300 m**
from a fragment of Fengyun-1C. That fragment has been up there since January 2007. Its elements are fresh,
and its worst-case probability sits right on our escalation line.

**Moment B—the rocket body.** Next Thursday, CYMBAL-11 passes about **420 m** from a spent Soviet upper
stage launched in 1975—far more mass than a fragment. But by then the rocket body's elements will be
nearly eight days old. **The honest recommendation is not "burn." It is "this is the one to watch; ask for
fresh tracking before you spend propellant on it."**

*These two paragraphs describe snapshot `20260925T0137Z`, and the cell above checks that the run reproduced
them. Refresh the snapshot and they must be rewritten from that cell's output.*

### How much do the assumptions move the answer?

Section 1 made three assumptions: a hard-body radius, a position uncertainty, and an escalation line.
Here is moment A under different hard-body radii:

In [ ]:
if "moment_A" in R:
    d = R["moment_A"]["miss_m"]
    sens = pd.DataFrame([{"hard_body_radius_m": hbr, "max_pc": max_pc(d, hbr), "pc_sigma_200m": pc_sigma(d, 200.0, hbr),
                          "pc_sigma_1km": pc_sigma(d, 1000.0, hbr), "escalate_on_worst_case": max_pc(d, hbr) >= ESCALATE_PC}
                         for hbr in (2.0, 3.0, 5.0, 10.0)])
    R["moment_A_sensitivity"] = sens.to_dict("records"); print(sens.to_string(index=False))

**At 5 m it escalates; at 3 m it does not.** Nothing about the orbits changed—only a number we chose.
That is exactly why an honest system shows its assumptions next to its answer: *here is what I'm
assuming, here is how the answer moves if I'm wrong.* The one-in-ten-thousand line is ours, too. Real
operators set their own, and it is a policy choice, not a physical constant.

**Two more caveats that belong on screen:**

- **Mean elements carry position errors that can run to kilometres**, growing with element age—far larger
  than a 300 m miss. A real operator treats a public-data screen as a *triage* step and requests
  higher-accuracy tracking (a conjunction data message, with covariance) before manoeuvring.
- **About 2,100 lost objects are not in this screen at all** (section 5.6). No news from them is not good news.

## 11—Validate before you build

Every check below reads the **loaded BigQuery tables**, not the dataframes in memory—because the agent
will read the tables. A **FAIL** means our pipeline is wrong and nothing downstream should be trusted. A
**WARN** means the source handed us something untidy, and the rows are printed.

In [ ]:
T = f"`{PROJECT}.{DATASET}"
v = q(f'''SELECT
  (SELECT COUNT(*) FROM {T}.catalog`) AS catalog_rows,
  (SELECT COUNT(DISTINCT NORAD_CAT_ID) FROM {T}.catalog`) AS catalog_ids,
  (SELECT COUNTIF(PERIGEE_KM > APOGEE_KM) FROM {T}.catalog`) AS perigee_above_apogee,
  (SELECT COUNTIF(PERIGEE_KM < 100) FROM {T}.catalog`) AS perigee_below_100km,
  (SELECT COUNT(*) FROM {T}.fleet`) AS fleet_rows,
  (SELECT COUNTIF(ABS(PERIGEE_KM - @alt) > 10 OR ABS(APOGEE_KM - @alt) > 10) FROM {T}.fleet`) AS fleet_off_altitude,
  (SELECT COUNT(*) FROM {T}.conjunctions`) AS conj_rows,
  (SELECT COUNTIF(miss_m >= 10000 OR miss_m < 0) FROM {T}.conjunctions`) AS conj_miss_out_of_range,
  (SELECT COUNTIF(max_pc < pc_sigma_200m OR max_pc < pc_sigma_1km OR max_pc > 1) FROM {T}.conjunctions`) AS pc_inconsistent,
  (SELECT COUNT(*) FROM {T}.conjunctions` c LEFT JOIN {T}.catalog` k USING (NORAD_CAT_ID)
     WHERE k.NORAD_CAT_ID IS NULL) AS conj_objects_not_in_catalog,
  (SELECT COUNT(*) FROM {T}.conjunctions` c LEFT JOIN {T}.fleet` f ON c.fleet_sat = f.OBJECT_NAME
     WHERE f.OBJECT_NAME IS NULL) AS conj_fleet_not_in_fleet,
  (SELECT COUNTIF(IS_STARLINK AND ON_ORBIT AND PERIGEE_KM <= @hi AND APOGEE_KM >= @lo) FROM {T}.satcat`) AS starlink_in_band
''', alt=FLEET_ALT_KM, lo=BAND[0], hi=BAND[1]).iloc[0]
print(v.to_string())
check("catalog holds every row we loaded", v.catalog_rows == len(CAT), f"{v.catalog_rows} vs {len(CAT)}")
check("one row per object in catalog", v.catalog_ids == v.catalog_rows, v.catalog_ids)
check("no orbit has perigee above apogee", v.perigee_above_apogee == 0, v.perigee_above_apogee)
note("objects with perigee below 100 km (re-entering now)", v.perigee_below_100km == 0,
     q(f"SELECT NORAD_CAT_ID, OBJECT_NAME, ROUND(PERIGEE_KM,1) AS perigee_km FROM {T}.catalog` WHERE PERIGEE_KM < 100 LIMIT 10").to_dict("records")
     if v.perigee_below_100km else 0)
check("twelve Cymbal satellites, all at our altitude", v.fleet_rows == 12 and v.fleet_off_altitude == 0, f"{v.fleet_rows} rows, {v.fleet_off_altitude} off")
check("the screen found approaches", v.conj_rows > 0, v.conj_rows)
check("every miss distance is inside the 10 km screen", v.conj_miss_out_of_range == 0, v.conj_miss_out_of_range)
check("every worst-case probability bounds its assumed-covariance ones", v.pc_inconsistent == 0, v.pc_inconsistent)
check("every conjunction joins to the catalog", v.conj_objects_not_in_catalog == 0, v.conj_objects_not_in_catalog)
check("every conjunction joins to the fleet", v.conj_fleet_not_in_fleet == 0, v.conj_fleet_not_in_fleet)
note("no Starlink satellite crosses our altitude", v.starlink_in_band == 0, v.starlink_in_band)
R["validation_raw"] = {k: (int(x) if pd.notna(x) else None) for k, x in v.items()}

## 12—Framing the agent

The notebook stops here, deliberately. Everything above is data engineering; everything below is
judgment, and judgment is the agent's job. `agent/` holds the working solution.

**What an operator asks between now and next Thursday, and what answers each:**

| The operator asks | What answers it |
|---|---|
| *"What should I worry about this week?"* | `conjunctions`, ranked, with the triage and the reasons—a query, then judgment about which reasons matter |
| *"What exactly is that object?"* | `catalog` and `satcat`: type, owner, launch year, whether it can dodge, which event made it |
| *"How confident are you?"* | The assumptions: hard-body radius, covariance, element age. The honest answer names all three |
| *"What if we burn 0.4 m/s prograde on Wednesday?"* | **Code the agent writes and runs in the sandbox**—propagate CYMBAL-11 with the burn applied and re-measure the miss |
| *"What does that cost us?"* | Propellant is mission life. The conversion needs a stated propellant budget, which is the operator's, not ours |
| *"Why not just dodge everything?"* | The week's numbers: hundreds of approaches under 10 km, a handful worth a human |
| *"What can't you see?"* | Lost objects, the missing covariance, and the error in mean elements |

**The output artifact is a Conjunction Assessment & Maneuver Recommendation:** which object, when, how
close, what the risk is under stated assumptions, what a manoeuvre would cost, and what we are
assuming. For moment B, the honest recommendation is *escalate, request fresh tracking, and have a burn
ready*—not *burn*.

In [ ]:
INFO = pd.DataFrame([{"snapshot": SNAPSHOT, "snapshot_uri": SNAPSHOT_URI, "gp_fetched_utc": MANIFEST["gp_full.csv"]["fetched_utc"],
                      "screen_start_utc": SCREEN_START, "screen_days": SCREEN_DAYS, "step_s": STEP_S,
                      "fleet_operator": "Cymbal Orbital (fictional)", "fleet_alt_km": FLEET_ALT_KM, "fleet_inclination_deg": INC,
                      "hbr_m_assumed": HBR_M, "sigma_m_assumed": ",".join(str(int(s)) for s in SIGMAS_M),
                      "escalate_max_pc": ESCALATE_PC, "watch_m": WATCH_M, "stale_days": STALE_DAYS, "citation": CITATION.strip()}])
n_info = load(INFO, "snapshot_info")
print(INFO.T.to_string(header=False))

## Appendix A—Maintainer only: publish the tables

`scripts/load.sh` rebuilds these tables without the notebook, from Parquet in Cloud Storage. This cell
writes that Parquet, by exporting the tables this run just built—so the fallback cannot drift from what
the notebook produces. It needs write access to the class bucket; **leave `PUBLISH = False`** unless you
are refreshing the snapshot.

In [ ]:
if PUBLISH:
    fatal = [c for c in CHECKS if c[0] == "FAIL"]
    assert not fatal, f"Not publishing over {len(fatal)} FAIL checks."
    dest = f"gs://class-demo/a4i-2026/demo-orbital-conjunction/tables/{SNAPSHOT}"
    for t in ("catalog", "satcat", "fleet", "conjunctions", "snapshot_info"):
        bq.extract_table(f"{PROJECT}.{DATASET}.{t}", f"{dest}/{t}/data.parquet",
                         job_config=bigquery.ExtractJobConfig(destination_format="PARQUET")).result()
        print("published", t)
    R["published"] = dest
else:
    print("PUBLISH is False—nothing written. (That is the right setting for everyone but a maintainer.)")

## Appendix B—Diagnostic summary

One block, everything a coach—or Patrick—needs. Copy all of it.

In [ ]:
R.update({"snapshot": SNAPSHOT, "screen_start": SCREEN_START.isoformat(), "tables": {"catalog": n_cat, "satcat": n_sat,
          "fleet": n_fleet, "conjunctions": n_conj, "snapshot_info": n_info}, "timings": TIMINGS,
          "checks": [" | ".join(c) for c in CHECKS],
          "checks_summary": {x: sum(1 for c in CHECKS if c[0] == x) for x in ("PASS", "FAIL", "OK", "WARN")},
          "top5": CONJ.head(5)[["fleet_sat", "object_name", "tca_utc", "miss_m", "max_pc", "triage"]].astype(str).to_dict("records")})
print("===== A4I DEMO 01 — DIAGNOSTIC BLOCK =====")
print(json.dumps(R, indent=1, default=str))
print("===== END =====")